# 06 誤差の分析

`04_train` が保存したモデル（`models/cloud_base.joblib`）の誤差を、**検証の年（2024）** の、正解が上限（1,524 m）より下の行について、
分けて数える。結果は `reports/error_analysis.json` に保存する（`docs/spec.md`「9. モデルの改善」、機能 #16）。

- 目的：「モデルが概算式に負けているのはどこか」を数値で出す（成功基準 2 は、上限より下の行の MAE を概算式と比べる）。
  分かるのは **どこで誤差が大きいか** まで。なぜそうなるかは、ここからは分からない。
- モデルは学習しない・変えない（読み込むだけ）。**テストの年（2025）は、読み込んだ直後に落とす**（ここでは見ない）。
- 比べるのは 2 つ：`model` と `rule_of_thumb`（概算式。Open-Meteo の気温・露点）。
- 分け方：都市、正解の高さの帯（`0_300`・`300_600`・`600_1000`・`1000_1524` m。境目は下の帯）、季節（現地の月）、
  現地の時の帯（夜 0〜5、朝 6〜11、午後 12〜17、夕 18〜23）、1 時間前からの正解の変化。
- 誤差の向き：`at_cap`（予測が上限 ＝ 低い天井を見逃した）、`too_high`（上限より下で、正解より高い）、`too_low`（正解より低い）、`exact`。
  偏り（`bias_below_cap_m`）は `予測 − 正解` の平均（正なら高すぎ）。
- 検証の年は、木の数と学習期間を決めるのに使っているので、誤差は少し良く出やすい。細かく分けた組は行が少ないので、行数を必ず見る。
- 集計の中身は `ml/error_analysis.py` にある（テスト済み）。このノートブックは順に呼ぶだけ。図は出さない。
- 既定は **サンプル**（`tests/fixtures/data_raw/` の 2 日ぶん。その場で小さく学習して、動くことだけを確かめる。数値に意味はない。一時的な場所に書く）。
  コミットしてある `reports/error_analysis.json` は、環境変数 `DATA_FETCH_MODE=full` を付けて全期間で実行したもの。外部とは通信しない。

出典：Weather data by Open-Meteo.com（CC BY 4.0）、Iowa Environmental Mesonet（Iowa State University）。

In [ ]:
import json
import os
import sys
import tempfile
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from app.cities import City
from ml import (
    archive,
    baseline,
    dataset,
    error_analysis,
    features,
    metar,
    model,
    raw_store,
    split,
    target,
    train,
)

## 0. 設定・読み込み

In [ ]:
MODE = os.environ.get("DATA_FETCH_MODE", "sample")

if MODE == "full":
    OUTPUT = ROOT / "reports" / "error_analysis.json"
    SPLIT = split.VALIDATION
    YEARS = list(split.SPLIT_YEARS[split.VALIDATION])
elif MODE == "sample":
    OUTPUT = Path(tempfile.mkdtemp()) / "error_analysis.json"
    # fixture は 2025 年の 2 日ぶんだけなので、分割で絞らずに、すべての行を見る。
    SPLIT = "sample"
    YEARS = [2025]
else:
    raise ValueError("DATA_FETCH_MODE must be 'sample' or 'full'")

pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", "{:.2f}".format)
print(f"mode={MODE}", SPLIT, YEARS)

In [ ]:
if MODE == "full":
    observations = dataset.load_dataset(ROOT / "data" / "processed" / "observations.parquet")
    dataset.validate_dataset(observations)
    # テストの年は、ここで落とす（分析でも見ない）。
    is_test = split.rows_in_years(observations, split.SPLIT_YEARS[split.TEST])
    observations = observations[~is_test].reset_index(drop=True)
    # 見るのは検証の年だけ（学習の年は、学習に使った行なので誤差が小さく出る）。
    observations = observations[split.rows_in_years(observations, YEARS)].reset_index(drop=True)
else:
    raw_root = ROOT / "tests" / "fixtures" / "data_raw"
    parts = {}
    for city in City:
        weather_path = raw_store.raw_path(raw_root, "open_meteo_archive", city, 2025, "json")
        metar_path = raw_store.raw_path(raw_root, "iem", city, 2025, "csv")
        parts[city.value] = (
            archive.parse_archive(weather_path.read_bytes().decode("utf-8")),
            metar.parse_metar(metar_path.read_bytes().decode("utf-8")),
        )
    observations = dataset.build_dataset(parts)
    dataset.validate_dataset(observations)

print(observations.shape, observations["time"].min(), observations["time"].max())

## 1. 特徴量・目的変数・予測

In [ ]:
X = features.features(observations)
y = target.target_m(observations)

if MODE == "full":
    cloud_base = model.load(ROOT / "models" / "cloud_base.joblib")
else:
    # `04_train` の出力は読まずに、同じ行でその場で学習する（動くことの確認だけ）。
    cloud_base = model.CloudBaseModel(booster=train.fit(X, y, X, y), train_years=(2025,))

predictions = {
    "model": model.predict_m(cloud_base, X),
    "rule_of_thumb": baseline.baseline_m(
        observations["temperature_c"], observations["dew_point_c"]
    ),
}
print(cloud_base.model_version, cloud_base.train_years)
pd.DataFrame(predictions).describe().T

## 2. 上限より下の行の誤差：都市・帯・季節・向き

`n_*` は行数、`abs_error_*_m` は誤差（絶対値）の合計。向きごとの合計を足すと、MAE × 行数になる。

In [ ]:
results = error_analysis.report(observations, y, predictions)


def table(grouping, names):
    """`grouping` の集計を、1 行 ＝ 予測 × 組の表にする。"""
    rows = []

    def walk(label, group, value):
        if len(group) == len(names):
            rows.append({"prediction": label, **dict(zip(names, group, strict=True)), **value})
            return
        for key, inner in value.items():
            walk(label, (*group, key), inner)

    for label, part in results.items():
        walk(label, (), part[grouping])
    return pd.DataFrame(rows)


COUNTS = ["n_below_cap", "mae_below_cap_m", "bias_below_cap_m"]
overall = pd.DataFrame({label: part["overall"] for label, part in results.items()}).T
overall

In [ ]:
table("by_city", ["city"]).set_index(["city", "prediction"]).sort_index()

In [ ]:
# 都市 × 帯：モデルと概算式の MAE・偏りを並べる。
by_band = table("by_city_band", ["city", "band"])
by_band.pivot_table(index=["city", "band"], columns="prediction", values=COUNTS, sort=False)

In [ ]:
# 都市 × 帯：モデルの向きごとの行数と、誤差の合計。
by_band[by_band["prediction"] == "model"].drop(columns=["prediction", *COUNTS[1:]]).set_index(
    ["city", "band"]
)

In [ ]:
by_season = table("by_city_season", ["city", "season"])
by_season.pivot_table(index=["city", "season"], columns="prediction", values=COUNTS, sort=False)

In [ ]:
# 都市 × 季節 × 帯：行の少ない組がある（`n_below_cap` を見る）。
by_season_band = table("by_city_season_band", ["city", "season", "band"])
by_season_band.pivot_table(
    index=["city", "season", "band"], columns="prediction", values=COUNTS, sort=False
)

## 3. 現地の時の帯、1 時間前からの正解の変化

「1 時間前からの変化」は、正解（METAR）の 1 時間前の値と比べたもの。分析のためだけの分け方で、特徴量にはできない（推論のときにない）。

In [ ]:
by_hour = table("by_city_hour_band", ["city", "hour_band"])
by_hour.pivot_table(index=["city", "hour_band"], columns="prediction", values=COUNTS, sort=False)

In [ ]:
by_change = table("by_city_change", ["city", "change"])
by_change.pivot_table(index=["city", "change"], columns="prediction", values=COUNTS, sort=False)

## 4. 保存

実行した時刻は入れない（再実行しても同じファイルになる）。

In [ ]:
report = {
    "mode": MODE,
    "model_version": cloud_base.model_version,
    "cap_m": target.CEILING_CAP_M,
    "split": SPLIT,
    "years": YEARS,
    "rows": "rows with target_m below cap_m",
    "error_sign": "prediction - target (positive: too high)",
    "height_bands_m": list(error_analysis.HEIGHT_BANDS),
    "predictions_note": {
        "model": "LightGBM (models/cloud_base.joblib), clipped to [0, cap_m]",
        "rule_of_thumb": "baseline: 125 * (temperature_c - dew_point_c) with Open-Meteo values",
    },
    "predictions": results,
}

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
text = json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False) + "\n"
OUTPUT.write_text(text, encoding="utf-8", newline="\n")
assert json.loads(OUTPUT.read_text(encoding="utf-8")) == report
print(OUTPUT.name, len(text))